In [1]:
import os
import sys
sys.path.insert(0,'/root/eallmancy')
sys.path.insert(0,'/root/eallmancy/experiments')
os.chdir('/root/eallmancy')

# TODO: THIS IS ULTRA HACKY, PUT API SETTINGS IN CONTROL PANEL
with open('/root/openai_api_key') as f:
    os.environ['OPENAI_API_KEY'] = f.read().replace('\n','')
with open('/root/gemini_api_key') as f:
    os.environ['GEMINI_API_KEY'] = f.read().replace('\n','')

from eallmancy.engine.llm_solution_engine_v3 import EngineV3
from eallmancy import Artifact
from eallmancy import SinglePartSolution, ProblemSpec
from eallmancy.engine.engine_trace import EngineTrace
from collections import defaultdict
import datetime
import numpy as np
import os
import re




In [2]:
from eallmancy.analysis.analyzer.utils import get_codet5p_embeddings, get_openai_embeddings, cosine_sim
from eallmancy.analysis.analyzer.analyzer import Analyzer

class PureEmbeddingAnalysis:
    def __init__(self, trace, embedding_func):
        all_solns = [a.trace_id for a in trace.list if a.is_solution]
        embs = embedding_func([CodeT5PAnalysis.get_clean_program(trace[ti]) for ti in all_solns])
        self.embs = {
            ti: emb for ti, emb in zip(all_solns, embs)
        }

    @classmethod
    def get_clean_program(cls, arte):
        return cls._clean_program(arte.program)
        
    @classmethod
    def _clean_program(cls, prog):
        # find function line
        fsig_idx = prog.find('def ')
        programbody_idx = fsig_idx + prog[fsig_idx:].find('\n')+1
        return prog[programbody_idx:]
        
    def distance(self, a, b):
        return 1 - cosine_sim(self.embs[a.trace_id], self.embs[b.trace_id])
        
class CodeT5PAnalysis(PureEmbeddingAnalysis):
    def __init__(self, trace):
        super().__init__(trace, embedding_func=get_codet5p_embeddings)
        
class OpenAIAnalysis(PureEmbeddingAnalysis):
    def __init__(self, trace):
        super().__init__(trace, embedding_func=get_openai_embeddings)

In [3]:
%%time

from pathlib import Path
import re
import pickle as pk

TRACEVAULT_DIR = '/root/data/trace_vault'

def find_or_default(ptrn, tstr, flags=0, default=None):
    return (re.findall(ptrn, tstr, flags=flags) or [default]).pop()



def fetch_trace(vault_name, pk_filename, max_traceid=float('inf')):
    trace_fp = Path('/root/data/trace_vault') / vault_name / pk_filename
    trace = EngineTrace.load(trace_fp.__str__())
    trace.trace_list = [a for a in trace.trace_list if a.trace_id<=max_traceid]
    trace.trace_graph = trace.trace_graph.subgraph([a.trace_id for a in trace.trace_list])
    for arte in trace.list:  # handle fudged solutions, scraped from reevo or llm4ad
        if arte.is_solution:
            if not hasattr(arte, 'program') and hasattr(arte, 'content'):
                arte.program = find_or_default(r'```[^\n]*\n(.*?)```', arte.content , flags=re.DOTALL, default=arte.content )
    return trace

def fetch_traceanalyses(vault_name, pk_filename):
    traceanalysis_fp = Path('/root/data/cached_analyses') / vault_name / pk_filename
    with open(traceanalysis_fp.__str__(), 'rb') as f:
        all_ta = pk.load(f)
    return all_ta['elmcy'], all_ta['codet5p'], all_ta['openai']

def precache_traceanalyses(vault_name, ignore_newerthan=None, max_traceid=float('inf'), segm_analysis=False):
    tracevault_dir = Path(TRACEVAULT_DIR) / vault_name
    traceanalysis_dir = Path('/root/data/cached_analyses') / vault_name
    
    for tid, trace_path in enumerate(tracevault_dir.glob('*.pk')):
        pk_filename = trace_path.name
        print(pk_filename)
        problem = find_or_default(r'_([^_,^\.]+)\.pk', pk_filename, default='')
        analyses_path = (traceanalysis_dir / pk_filename)
        try:
            trace = EngineTrace.load(trace_path)
        except:
            print('    failed to load, skipping')
            continue
        trace.trace_list = [a for a in trace.trace_list if a.trace_id <= max_traceid]
        for arte in trace.list:  # handle fudged solutions, scraped from reevo or llm4ad
            if arte.is_solution:
                if not hasattr(arte, 'program') and hasattr(arte, 'content'):
                    arte.program = find_or_default(r'```[^\n]*\n(.*?)```', arte.content , flags=re.DOTALL, default=arte.content )
    
        if ignore_newerthan is not None and analyses_path.exists() and datetime.datetime.strptime(ignore_newerthan, '%Y-%m-%d_%H:%M:%S').timestamp() < os.path.getmtime(analyses_path):
            print('    skipping',pk_filename,'base analysis recalc')
            with open(analyses_path.__str__(), 'rb') as f:
                analyses = dict(**{
                    k:v for k,v in
                    zip(['elmcy', 'codet5p', 'openai'], fetch_traceanalyses(vault_name, pk_filename))
                })
        else:
            try:
                analyses = dict(
                    elmcy=Analyzer.build_pairwise_analysis(trace, trace.problem_spec),
                    codet5p=CodeT5PAnalysis(trace),
                    openai=OpenAIAnalysis(trace),
                )
                
                with open(analyses_path.__str__(), 'wb') as f:
                    pk.dump(analyses, f)
            except:
                print('\t failed for some reason...')
                continue
            
        segm_analyses_path = (Path('/root/data/cached_segm_analyses') / vault_name / pk_filename)
        if segm_analysis:
            if segm_analyses_path.exists() and datetime.datetime.strptime(ignore_newerthan, '%Y-%m-%d_%H:%M:%S').timestamp() < os.path.getmtime(segm_analyses_path):
                print('skipping', segm_analyses_path.__str__())
                segm_ta = None
            else:
                # similar process
                print('  building segment analysis', segm_analyses_path.__str__())
                segm_analyses_path.parent.mkdir(exist_ok=True, parents=True)
                segm_ta = (
                    Analyzer(trace, cached_tcmp=analyses['elmcy']._tcmp, invalid_solns=analyses['elmcy']._invalid_solns)
                    .segment_analysis(
                        emb_cutoff=0.9,
                        min_members=2, min_tokens=12,
                        verbose=False,
                    )
                )
                segm_ta._trace.lock = None
                with open(segm_analyses_path.__str__(), 'wb') as f:
                    pk.dump(segm_ta, f)
        else:
            segm_ta = None

        del analyses, segm_ta # free up mem

# took 1h in total for og_benchmark
# precache_traceanalyses('og_benchmark', ignore_newerthan='2025-08-30_16:00:00')
precache_traceanalyses('elmcy_reevo', ignore_newerthan='2025-08-31_16:00:00', segm_analysis=True)
# precache_traceanalyses('elmcy_eoh', ignore_newerthan='2025-08-30_20:00:00', segm_analysis=True)
precache_traceanalyses('divrs_fs',    ignore_newerthan='2025-08-31_16:00:00', max_traceid=150, segm_analysis=True)
# precache_traceanalyses('meta_eoh', ignore_newerthan='2025-08-31_16:00:00', segm_analysis=True)

# precache_traceanalyses('gmni_reevo', ignore_newerthan='2025-08-30_16:00:00', segm_analysis=True)
# precache_traceanalyses('gmni_fs', ignore_newerthan='2025-08-30_16:00:00', segm_analysis=False)


20250802-235310_a266_reevo_op_aco.pk
    skipping 20250802-235310_a266_reevo_op_aco.pk base analysis recalc
  building segment analysis /root/data/cached_segm_analyses/elmcy_reevo/20250802-235310_a266_reevo_op_aco.pk


<timed exec>:89: DeprecationWarning: Pickle, copy, and deepcopy support will be removed from itertools in Python 3.14.


20250803-010854_7f89_reevo_op_aco.pk
    skipping 20250803-010854_7f89_reevo_op_aco.pk base analysis recalc
  building segment analysis /root/data/cached_segm_analyses/elmcy_reevo/20250803-010854_7f89_reevo_op_aco.pk
20250803-013958_8849_reevo_tsp_aco.pk
    skipping 20250803-013958_8849_reevo_tsp_aco.pk base analysis recalc
  building segment analysis /root/data/cached_segm_analyses/elmcy_reevo/20250803-013958_8849_reevo_tsp_aco.pk
20250802-235352_c6cb_reevo_op_aco.pk
    skipping 20250802-235352_c6cb_reevo_op_aco.pk base analysis recalc
  building segment analysis /root/data/cached_segm_analyses/elmcy_reevo/20250802-235352_c6cb_reevo_op_aco.pk
20250803-002048_663d_reevo_tsp_aco.pk
    skipping 20250803-002048_663d_reevo_tsp_aco.pk base analysis recalc
  building segment analysis /root/data/cached_segm_analyses/elmcy_reevo/20250803-002048_663d_reevo_tsp_aco.pk
20250803-010407_f23e_reevo_tsp_aco.pk
    skipping 20250803-010407_f23e_reevo_tsp_aco.pk base analysis recalc
  building segme

In [ ]:
{12}.

In [4]:
print(2)

2
